<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

# 🧠 Runtime Context 与「工具中转」式访问

**一句话定位**：context 是「这次调用属于哪个用户」的只读数据，它不会被塞进 prompt，而是靠 agent 自己调用工具去读取。



</div>

## 🤔 `context_schema` vs `context` —— 区别在哪

两者长得像，但角色完全不同：

| 参数 | 角色 | 内容 |
|---|---|---|
| `context_schema=ColourContext` | 📐 类型声明 | 只是类，没有数据 |
| `context=ColourContext()` | 📦 真实数据 | 一个带具体值的实例 |

> 💡 类比**函数签名 vs 函数调用**：

```python
# context_schema 相当于声明参数类型
def agent(context: ColourContext): ...

# context= 相当于传真实参数
agent(context=ColourContext(favourite_colour="green"))
```

`ColourContext`定义里写的默认值（如 `favourite_colour="blue"`）只是类定义上的兜底值。最后一个 cell 证明了这点：传 `context=ColourContext(favourite_colour="green")` 能正确覆盖默认值 —— 如果数据已经由 `context_schema` 提供了，这个覆盖根本不该生效。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**实战用法**：真实项目里 `context_schema` 通常是一个**没有默认值的空类**，每次调用时都用 `context=` 传入真实的、按用户区分的数据——**同一个 agent，不同用户，每次调用传不同的 context**。

</div>

---

## 🔧 Context 是怎么传到 LLM 手里的

![Context via Tool Calls](resources/colour_context_tool_runtime.svg)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：`context_schema` **不会**被直接塞给 LLM。LLM 对 context 对象本身是**完全不可见**的。

</div>

Context 是**通过工具调用间接暴露**的：
1. 🤖 LLM 想知道用户最喜欢的颜色
2. 🔨 它调用工具 `get_favourite_colour`
3. ⚙️ 工具从 `ToolRuntime` 里读出值并返回
4. 💬 LLM 把这个结果当作 tool message 收到

---

## ❓ 为什么不直接把 context 塞进 system prompt

简单回答：**context 很容易变得很大** 📈

真实应用里，用户的 context 对象可能包含完整的用户画像、偏好、历史记录、权限、账户状态等等。把这些全部塞进每次 prompt 会带来实际成本：

| 问题 | 影响 |
|---|---|
| 🪙 **Token 开销** | 挤占 context window，推理空间变小 |
| 🌀 **分心** | "Lost in the middle" —— prompt 越臃肿，模型表现越差 |
| 🐢 **延迟与成本** | 输入 token 越多，响应越慢、账单越贵 |
| 🔒 **隐私** | LLM 看到了所有字段，包括它根本不需要知道的 |

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：工具调用模式用**懒加载、按需访问**解决了这个问题 —— LLM 只在真正需要时才去取它需要的那一小块数据。这和 **RAG**（Retrieval-Augmented Generation）背后的原理是一致的：不是把整个知识库提前塞进 prompt，而是查询时才检索相关片段。📚

</div>

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### 🐍 `@dataclass` 是什么

`dataclass` **不是** LangChain 的东西 —— 它是标准 Python（Python 3.7 起内置）。它是创建「主要用来装数据」的类的一个捷径：

```python
# 不用 dataclass —— 啰嗦
class ColourContext:
    def __init__(self, favourite_colour="blue"):
        self.favourite_colour = favourite_colour

# 用 dataclass —— 效果一样，代码更少
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
```

`@dataclass` 装饰器会根据带类型注解的字段自动生成 `__init__`、`__repr__`、`__eq__`。

LangChain 用 dataclass 来定义 `context_schema`，原因是：
- ✅ 轻量，是标准 Python，不引入额外依赖
- ✅ 类型注解能告诉框架数据的形状
- ✅ 默认值在开发/测试时很方便
- ✅ 按惯例是只读的 —— 契合 context「只读」的本质

> 💡 对比一下 `AgentState`（用于 `state_schema`）—— 它继承自 LangGraph 的类，不是 dataclass，因为 state 需要支持可变更和 checkpointer 集成。

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='0a8fb4d8-6147-4e75-9e97-c7702363cbb3'),
              AIMessage(content='I don’t know your favorite color yet. Want me to guess, or help you figure it out?\n\nIf you’d like a quick guess, blue is a common favorite—is that right? If not, tell me a bit about your preferences (for example, warm or cool colors, bright or muted tones, and what you’ll use the color for), and I’ll suggest some you might love.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1178, 'prompt_tokens': 12, 'total_tokens': 1190, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1088, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZog978z3x9aUvm

## 🔧 访问 Context

下面用两个工具演示：agent 怎么通过 `ToolRuntime.context` 读取当前调用的 context 数据。

In [6]:
from langchain.tools import tool, ToolRuntime


@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

pprint(response)

/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='2a16b64a-8b8b-4854-b0ea-c5e08c7684fc'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZosmQ2f1IgEymjiD7CbvoFjWNcPi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd70c-f023-7f22-af88-8c7ed8aea275-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_OW7ttOVwGC8jRte3EZFe7H5h', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green"),
)

pprint(response)

/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='5220e357-64ee-413c-9150-1d36bf7f2883'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZp0FSyE96NhiCVPRkvm4w6AVnvpj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd714-0690-7e22-b5c7-f6192acbea9e-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_OIuGVlf2vbGflkiNiezxD3Sr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_